# AI Inventory Demand & Stock Risk Prediction
## 01 — Data Understanding

This notebook explores the historical sales and inventory dataset using PySpark. It checks the file structure, column types, missing values, duplicate rows, date coverage, and the number of stores and products.

The original CSV is kept under `data/raw/`. This notebook reads the data for analysis and does not modify the source file.

## Load the CSV with PySpark

The CSV is read into a Spark DataFrame. The header row supplies the column names, and `inferSchema` asks Spark to infer data types for this initial exploration.

The notebook is inside the `notebooks/` folder, so its file path must point up one level to the repository root and then into `data/raw/`.


## Inspect the dataset structure

This section displays the column names and inferred data types, counts the records, and shows five sample rows. The sample helps us check that dates, categories, and numeric values look as expected.

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("inspect-sales-data")
    .getOrCreate()
)

try:
    sales = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("../data/raw/sales_data.csv")
    )

    print("Columns and data types:")
    sales.printSchema()

    print("Row count:", sales.count())

    print("First five rows:")
    sales.show(5, truncate=False)

finally:
    spark.stop()

Columns and data types:
root
 |-- Date: date (nullable = true)
 |-- Store ID: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Inventory Level: integer (nullable = true)
 |-- Units Sold: integer (nullable = true)
 |-- Units Ordered: integer (nullable = true)
 |-- Price: double (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Weather Condition: string (nullable = true)
 |-- Promotion: integer (nullable = true)
 |-- Competitor Pricing: double (nullable = true)
 |-- Seasonality: string (nullable = true)
 |-- Epidemic: integer (nullable = true)
 |-- Demand: integer (nullable = true)

Row count: 76000
First five rows:
+----------+--------+----------+-----------+------+---------------+----------+-------------+-----+--------+-----------------+---------+------------------+-----------+--------+------+
|Date      |Store ID|Product ID|Category   |Region|Inventory Level|Units Sold|Un

## Check data quality

This section counts missing values in each column and checks for fully duplicated rows. It also reports the date range and how many distinct stores and products are present.

These checks help identify basic data issues before feature engineering or model training. They do not change or remove any records.

In [7]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("inspect-sales-data")
    .getOrCreate()
)

try:
    sales = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("../data/raw/sales_data.csv")
    )

    print("Columns and data types:")
    sales.printSchema()

    print("Row count:", sales.count())

    print("First five rows:")
    sales.show(5, truncate=False)


    from pyspark.sql import functions as F

    null_counts = sales.select([
        F.sum(F.col(f"`{column}`").isNull().cast("int")).alias(column)
        for column in sales.columns
    ]).first().asDict()

    print("Missing values per column:", null_counts)
    print("Duplicate rows:", sales.count() - sales.dropDuplicates().count())

    date_range = sales.select(
        F.min("Date").alias("first_date"),
        F.max("Date").alias("last_date")
    ).first()

    print("Date range:", date_range["first_date"], "to", date_range["last_date"])
    print("Unique stores:", sales.select("Store ID").distinct().count())
    print("Unique products:", sales.select("Product ID").distinct().count())


finally:
    spark.stop()

Columns and data types:
root
 |-- Date: date (nullable = true)
 |-- Store ID: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Inventory Level: integer (nullable = true)
 |-- Units Sold: integer (nullable = true)
 |-- Units Ordered: integer (nullable = true)
 |-- Price: double (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Weather Condition: string (nullable = true)
 |-- Promotion: integer (nullable = true)
 |-- Competitor Pricing: double (nullable = true)
 |-- Seasonality: string (nullable = true)
 |-- Epidemic: integer (nullable = true)
 |-- Demand: integer (nullable = true)

Row count: 76000
First five rows:
+----------+--------+----------+-----------+------+---------------+----------+-------------+-----+--------+-----------------+---------+------------------+-----------+--------+------+
|Date      |Store ID|Product ID|Category   |Region|Inventory Level|Units Sold|Un